In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier

import optuna

In [27]:
df = pd.read_csv("datasets/kingametric_credit_risk.csv")

In [28]:
df.shape

(8744, 45)

In [31]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [32]:
cat_cols = ["Payment_of_Min_Amount", "Credit_Mix", "Payment_Behaviour", "Borrower_Tier"]

df[cat_cols] = df[cat_cols].astype("category")

In [33]:
X = df.drop(columns=["Default_Flag"], axis=1)
y = df["Default_Flag"]

In [34]:
X.shape

(8744, 44)

In [36]:
corr = X.corr(numeric_only=True).abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]

X = X.drop(columns=to_drop)

In [37]:
X.shape

(8744, 35)

In [38]:
folds = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

In [39]:
base_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    enable_categorical=True, 
    tree_method="hist",
    learning_rate=0.05,
    random_state=42
)
base_model.fit(X, y)

importances = base_model.feature_importances_

feature_importance = (
    pd.Series(importances, index=X.columns).sort_values(ascending=False)
)

In [40]:
top_features = feature_importance.head(20).index

X_select = X[top_features]

In [43]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.7, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.95),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 5),
        "eval_metric": "auc",
        "tree_method": "hist",
        "enable_categorical":True,
        "random_state": 42
    }

    auc_scores = []

    for train_idx, val_idx in folds.split(X_select, y):
        
        X_train, X_val = X_select.iloc[train_idx], X_select.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBClassifier(**params,)

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:,1]

        auc = roc_auc_score(y_val, y_pred)
        auc_scores.append(auc)

    return np.mean(auc_scores)

In [44]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=40)

[I 2026-03-22 17:50:37,199] A new study created in memory with name: no-name-cf953a8e-1336-42b7-8b76-023cf8237355
[I 2026-03-22 17:50:40,002] Trial 0 finished with value: 0.6121729632982871 and parameters: {'n_estimators': 586, 'max_depth': 6, 'learning_rate': 0.0630816411452695, 'subsample': 0.7591267342787882, 'colsample_bytree': 0.7009985062469566, 'min_child_weight': 9, 'gamma': 0.1007118391959288, 'reg_alpha': 0.06783587471716501, 'reg_lambda': 1.5150455363431174, 'scale_pos_weight': 3.2577783900742205}. Best is trial 0 with value: 0.6121729632982871.
[I 2026-03-22 17:50:43,261] Trial 1 finished with value: 0.6111543371383699 and parameters: {'n_estimators': 768, 'max_depth': 5, 'learning_rate': 0.059996457190343466, 'subsample': 0.7640449748598989, 'colsample_bytree': 0.791721846094584, 'min_child_weight': 1, 'gamma': 0.48224216516532337, 'reg_alpha': 0.2229080041984508, 'reg_lambda': 2.013969158408585, 'scale_pos_weight': 4.16891237016034}. Best is trial 0 with value: 0.61217296

In [45]:
best_params = study.best_params

print(best_params)

{'n_estimators': 338, 'max_depth': 4, 'learning_rate': 0.010639765970054421, 'subsample': 0.8072754951374619, 'colsample_bytree': 0.8536659630541316, 'min_child_weight': 6, 'gamma': 0.14599220599953236, 'reg_alpha': 0.6240138684600905, 'reg_lambda': 2.634026186223532, 'scale_pos_weight': 3.0825670049983698}


In [46]:
fin_model = XGBClassifier(**best_params, enable_categorical=True, tree_method="hist")

fin_model.fit(X_select, y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8536659630541316
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None
